In [23]:
import os
import numpy as np

# === Config ===
input_file_path = 'reversed_synthetic_dataset_n100000000_k5.txt'
use_char_encoding = True      # True = character-level, False = tiktoken BPE
reversed = False       # True = reverse each line (e.g. for BW training)

# === Load and optionally reverse lines ===
if not os.path.exists(input_file_path):
    raise FileNotFoundError(f"{input_file_path} not found.")

with open(input_file_path, 'r', encoding='utf-8') as f:
    lines = f.read().splitlines()

if reversed:
    print("Reversing each line character-wise...")
    lines = [line[::-1] for line in lines]


In [24]:
data = ''.join(lines)
line_len   = len(lines[0])           # all lines equal
split_rows = int(num_samples * 0.9)                       # 90 % of rows
split_idx  = split_rows * line_len                      # multiple of L

train_data = data[:split_idx]
val_data   = data[split_idx:]

In [26]:
# === Encoding ===
base_chars     = sorted(set(data) - {' '})   # all “normal” chars
# arrow_tokens   = [f'↔_{i}' for i in range(7)]
# cross_tokens   = [f'×_{i}' for i in range(3)]
vocab          = base_chars #+ arrow_tokens + cross_tokens
# vocab

In [27]:
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
stoi

{'0': 0,
 '1': 1,
 '2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '×': 10,
 '↔': 11}

In [28]:
def encode(s: str):
    ids = []
    for ch in s:
        if ch == ' ':
            continue                          # skip spaces
        elif ch == '↔':
            ids.extend(stoi['↔'] for _ in range(7))
        elif ch == '×':
            ids.extend(stoi['×'] for _ in range(3))
        else:
            ids.append(stoi[ch])
    return ids

def decode(ids):
    """
    Collapse ↔_* → ↔ and ×_* → × while reconstructing the string.
    (If you don’t need decoding, you can drop this and just map
     every placeholder back to its symbol.)
    """
    out, i = [], 0
    while i < len(ids):
        tok = itos[ids[i]]
        if tok=='↔':
            out.append('↔');  i += 7
        elif tok =='×':
            out.append('×');  i += 3
        else:
            out.append(tok);  i += 1
    return ''.join(out)

In [29]:
assert len(encode('↔')) == 7
assert len(encode('×')) == 3
assert len(encode(' ')) == 0

In [30]:
train_ids = encode(train_data)

In [31]:
print(train_ids[:41])

[7, 1, 7, 2, 7, 9, 3, 6, 3, 5, 11, 11, 11, 11, 11, 11, 11, 9, 4, 5, 4, 1, 10, 10, 10, 5, 6, 7, 3, 7, 1, 8, 9, 6, 7, 3, 6, 7, 3, 3, 11]


In [34]:
decode(train_ids[-41:])

'939×423731783881170↔82499×08629'

In [21]:
len(vocab)

20

In [2]:
import numpy as np
val = np.memmap('/Users/sophiasklaviadis/word-order/nanoGPT/data/prime/bw_100000000_char_val.bin', dtype=np.uint16, mode='r')
len(val)

300000000